# Suite CMP — Comparison

`Comparison.OfX` implements `Visitors.OfX` and records what is written into it; `a.compare(b)` returns -1, 0 or 1,
or `None` when the two are incomparable. Equality follows EQUALITY.md: natives by canonical value,
objects by their declared properties, links by identity, adjacency entries as sets. Ordering is defined only for
`int`, `float`, `str` and `bytes`.

In [ ]:
import math

from mbse.Schemas.Framework import Comparison as C, Proxies, Schemas as S
from support import get_property, raises, set_property, text

Int, Float, Str, Bool, Bytes = (S.OfNative.Data(t) for t in (int, float, str, bool, bytes))


def cmp(schema, a, b):
    """Compares two native values under `schema`."""
    return C.OfNative(schema, a).compare(C.OfNative(schema, b))


Tag = S.OfRelation.Builder().links("item", "tag").properties(text("note")).create()
Point = S.OfObject.Builder().properties(text("x", int), text("label")).relations(
    lambda r: r.name("tags").of(Tag).me("item")).create()
Label = S.OfObject.Builder().properties(text("name")).relations(lambda r: r.name("items").of(Tag).me("tag")).create()
for name, schema in [("Point", Point), ("Label", Label), ("Tag", Tag)]:
    Proxies.register(name, schema)
B = Proxies.Builders

## CMP-01 · Ordered natives: ints, strings by code point, bytes lexicographically

In [ ]:
assert cmp(Int, 1, 2) == -1 and cmp(Int, 2, 1) == 1 and cmp(Int, 2, 2) == 0
assert cmp(Int, 2**70, 2**70 + 1) == -1  # beyond 2**53
assert cmp(Str, "a", "b") == -1 and cmp(Str, "ab", "a") == 1 and cmp(Str, "", "") == 0
assert cmp(Str, "\uffff", "\U00010000") == -1  # by code point, not by UTF-16 code unit
assert cmp(Str, "e\u0301", "\u00e9") == -1  # no Unicode normalization
assert cmp(Bytes, b"\x01", b"\x02") == -1 and cmp(Bytes, b"\x01\x00", b"\x01") == 1 and cmp(Bytes, b"", b"") == 0
assert cmp(Bytes, b"\xff", b"\x01\x00") == 1

## CMP-02 · Floats: by value, `-0.0` before `0.0`, NaNs equal each other and nothing else

In [ ]:
nan, inf = math.nan, math.inf
assert cmp(Float, 1.5, 2.5) == -1 and cmp(Float, -inf, inf) == -1 and cmp(Float, 0.1 + 0.2, 0.3) == 1
assert cmp(Float, -0.0, 0.0) == -1 and cmp(Float, 0.0, -0.0) == 1 and cmp(Float, -0.0, -0.0) == 0
assert cmp(Float, nan, nan) == 0 and cmp(Float, nan, -nan) == 0
assert cmp(Float, nan, 1.0) is None and cmp(Float, inf, nan) is None

## CMP-03 · Booleans are equal or incomparable

In [ ]:
assert cmp(Bool, True, True) == 0 and cmp(Bool, False, False) == 0
assert cmp(Bool, False, True) is None and cmp(Bool, True, False) is None

## CMP-04 · Absent values, wrong types and distinct native types

In [ ]:
absent = C.OfNative(Int)
assert not absent.has() and absent.compare(C.OfNative(Int)) == 0
assert absent.compare(C.OfNative(Int, 1)) is None and C.OfNative(Int, 1).compare(absent) is None
with raises(AttributeError, match="the value is not set"):
    absent.get()
value = C.OfNative(Int, 1)
assert value.get() == 1 and value.clear() is value and not value.has()
assert value.set(3) is value and value.get() == 3
with raises(TypeError, match="expected int, got bool"):
    C.OfNative(Int, True)
with raises(TypeError, match="expected float, got int"):
    C.OfNative(Float).set(1)
assert C.OfNative(Int, 1).compare(C.OfNative(Float, 1.0)) is None  # distinct native types are never equal
assert C.OfNative(S.OfNative.Data(int), 1).compare(C.OfNative(Int, 1)) == 0  # the same native type

## CMP-05 · Objects are equal when their declared properties are, and are otherwise incomparable

In [ ]:
p = B.Point().x(1).label("a").create()
unlabeled = B.Point().x(1).create()
assert C.OfObject(Point, p).compare(C.OfObject(Point, B.Point(p).clone())) == 0
assert C.OfObject(Point, p).compare(C.OfObject(Point, B.Point().x(2).label("a").create())) is None  # not ordered
assert C.OfObject(Point, p).compare(C.OfObject(Point, unlabeled)) is None  # present and absent
assert C.OfObject(Point, unlabeled).compare(C.OfObject(Point, B.Point().x(1).create())) == 0
assert C.OfObject(Point).compare(C.OfObject(Point)) == 0
assert C.OfObject(Point, p).compare(C.OfObject(Label, B.Label().name("a").create())) is None  # other schemas

## CMP-06 · Adjacencies do not take part in comparing objects

In [ ]:
t1, t2 = B.Label().name("t").create(), B.Label().name("t").create()
tagged = B.Point().x(1).label("a").tags(lambda x: x.tag(t1).note("n")).create()
assert C.OfObject(Point, p).compare(C.OfObject(Point, tagged)) == 0

## CMP-07 · Links compare by the identity of their targets

In [ ]:
link = C.OfLink("tag")
assert link.name() == "tag" and link.compare(C.OfLink("tag")) == 0
with raises(ValueError, match="link 'tag' is not set"):
    link.target(lambda t: None)
assert link.set(t1) is link and link.compare(C.OfLink("tag")) is None and C.OfLink("tag").compare(link) is None
assert link.compare(C.OfLink("tag").set(t1)) == 0
assert link.compare(C.OfLink("tag").set(t2)) is None  # equal content, different objects
seen = []
assert link.target(seen.append) is link and seen == [t1]

## CMP-08 · Entries are equal when their links and properties are

In [ ]:
def entry(tag=None, **values):
    """An entry seen from its `item`, linking `tag`."""
    recorded = C.OfEntry(Tag, "item")
    if tag is not None:
        recorded.link("tag", lambda k: k.set(tag))
    for name, value in values.items():
        set_property(recorded, name, value)
    return recorded


assert entry(t1, note="n").compare(entry(t1, note="n")) == 0
assert entry(t1, note="n").compare(entry(t2, note="n")) is None
assert entry(t1, note="n").compare(entry(t1, note="m")) is None
assert entry(t1).compare(entry(t1, note="n")) is None and entry(t1).compare(entry(t1)) == 0
assert entry(t1).compare(entry()) is None
assert entry(t1).compare(C.OfEntry(Tag, "tag")) is None  # seen from the other end
assert C.OfEntry(Tag).compare(C.OfEntry(Tag)) == 0  # no link implied, none set
Other = S.OfRelation.Builder().links("item", "tag").create()
assert C.OfEntry(Other, "item").compare(C.OfEntry(Tag, "item")) is None
with raises(KeyError, match="'item' is not a link this entry can set"):
    entry().link("item", lambda k: None)  # the object's own link
with raises(KeyError, match="unknown property 'nope'"):
    entry(nope="x")

## CMP-09 · An adjacency's entries form a set

In [ ]:
def tags(point):
    """The `tags` adjacency recorded from `point`."""
    found = []
    C.OfObject(Point, point).adjacency("tags", found.append)
    return found[0]


both = B.Point().x(1).tags(lambda x: x.tag(t1).note("n")).tags(lambda x: x.tag(t2)).create()
reversed_ = B.Point().x(1).tags(lambda x: x.tag(t2)).tags(lambda x: x.tag(t1).note("n")).create()
one = B.Point().x(1).tags(lambda x: x.tag(t1).note("n")).create()
different = B.Point().x(1).tags(lambda x: x.tag(t1).note("n")).tags(lambda x: x.tag(t1)).create()
assert tags(both).compare(tags(reversed_)) == 0  # order does not matter
assert tags(both).compare(tags(one)) is None and tags(both).compare(tags(different)) is None
assert tags(both).name() == "tags" and tags(both).me() == "item"
from_label = []
C.OfObject(Label, t1).adjacency("items", from_label.append)
assert tags(both).compare(from_label[0]) is None  # the same relation, seen from the other end
Elsewhere = S.OfObject.Builder().relations(lambda r: r.name("tags").of(Other).me("item")).create()
elsewhere = []
C.OfObject(Elsewhere).adjacency("tags", elsewhere.append)
assert tags(one).compare(elsewhere[0]) is None  # another relation

adjacency = tags(one)
assert adjacency.add(lambda e: (e.link("tag", lambda k: k.set(t1)), set_property(e, "note", "n"))) is adjacency
recorded = []
assert adjacency.entries(recorded.append) is adjacency and len(recorded) == 1  # the equal entry was elided
adjacency.add(lambda e: e.link("tag", lambda k: k.set(t2)))
recorded.clear()
adjacency.entries(recorded.append)
assert len(recorded) == 2 and adjacency.remove(recorded[0]) is adjacency
recorded.clear()
adjacency.entries(recorded.append)
assert len(recorded) == 1

## CMP-10 · Object-valued properties, and values of different kinds

In [ ]:
Inner = S.OfObject.Builder().properties(text("x", int)).create()
Box = S.OfObject.Builder().properties(lambda p: p.name("corner").of(lambda t: t.as_object(Inner))).create()


def box(x):
    """A box whose corner has `x`, written through the visitor protocols (proxies hold only natives)."""
    recorded = C.OfObject(Box)
    recorded.property("corner", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "x", x))))
    return recorded


assert box(1).compare(box(1)) == 0 and box(1).compare(box(2)) is None and box(1).compare(C.OfObject(Box)) is None
reused = box(1)
reused.property("corner", lambda p: p.value(lambda a: a.as_object(lambda o: set_property(o, "x", 2))))
assert reused.compare(box(2)) == 0  # the same recorded object, written twice
with raises(TypeError, match="the schema is not a native schema"):
    set_property(C.OfObject(Box), "corner", 1)
with raises(TypeError, match="the schema is not an object schema"):
    C.OfObject(Point).property("x", lambda p: p.value(lambda a: a.as_object(lambda o: None)))
native = C.OfProperty("v", Int).value(lambda a: a.as_native(lambda n: n.set(1)))
obj = C.OfProperty("v", Inner).value(lambda a: a.as_object(lambda o: None))
assert native.compare(obj) is None and obj.compare(native) is None

## CMP-11 · Recordings read back through the visitor protocols

In [ ]:
record = C.OfObject(Point, p)
names = []
assert record.properties(lambda q: names.append((q.name(), q.has()))) is record
assert names == [("x", True), ("label", True)]
assert record.has("x") and not C.OfObject(Point).has("x") and get_property(record, "label") == "a"
assert record.clear("label") is record and not record.has("label")
record.property("x", lambda q: q.clear())
names.clear()
record.properties(names.append)
assert not record.has("x") and names == []
seen = []
assert record.adjacencies(lambda a: seen.append(a.name())) is record and seen == ["tags"]
with raises(KeyError, match="unknown adjacency 'nope'"):
    record.adjacency("nope", lambda a: None)
with raises(KeyError, match="unknown property 'nope'"):
    record.property("nope", lambda q: None)
recorded = C.OfEntry(Tag, "item")
seen.clear()
assert recorded.links(lambda k: (seen.append(k.name()), k.set(t1))) is recorded
set_property(recorded, "note", "n")
assert recorded.has("note") and recorded.properties(lambda q: seen.append(q.name())) is recorded
assert seen == ["tag", "note"] and recorded.clear("note") is recorded and not recorded.has("note")

## CMP-12 · Union and intersection values cannot be compared yet

In [ ]:
for method in ["as_union", "as_intersection"]:
    with raises(NotImplementedError, match="values cannot be compared yet"):
        getattr(C.OfAny(Int), method)(lambda v: None)